# Source coding

> A payload as bits, and back: integers, codebook indices, quantized real numbers and vectors.

In [ ]:
#| default_exp source.codec

In [ ]:
#| hide
from nbdev.showdoc import *

Source coding turns a payload into bits and back: the first and last stages of a link-level transmission (see
`radio.link.chain`). Each piece is a plain function or a small class on numpy arrays or torch tensors (on any device),
with any batch dimensions in front, so training code can also use them on whole batches of links, without a
`Network`.

In [ ]:
#| export
from __future__ import annotations

import math
from abc import ABC, abstractmethod

import numpy as np

from comm_core.block import _is_torch

In [ ]:
#| export
def _arange(n: int, like):
    "0, ..., n-1: a torch tensor on `like`'s device if `like` is one, else a numpy array."
    if _is_torch(like):
        import torch
        return torch.arange(n, device=like.device)
    return np.arange(n)


def _as_array(x, dtype=None):
    "Tensors stay tensors; anything else becomes a numpy array (at least 1-d)."
    return x if _is_torch(x) else np.atleast_1d(np.asarray(x, dtype=dtype))

## Bits

Integers and their bits, least significant bit first. Both functions work on the last dimension.

In [ ]:
#| export
def to_bits(
    indices, # Integers, (..., M)
    bits_per_index: int, # Bits per integer
):
    "The bits of `indices`: (..., M * bits_per_index), least significant bit first."
    shifts = _arange(bits_per_index, indices)
    bits = (indices[..., None] >> shifts) & 1
    return bits.reshape(*indices.shape[:-1], -1)


def from_bits(
    bits, # 0/1 or booleans, (..., M * bits_per_index)
    bits_per_index: int, # Bits per integer
):
    "The integers of `bits` (the inverse of `to_bits`): (..., M)."
    weights = 2 ** _arange(bits_per_index, bits)
    return (bits.reshape(*bits.shape[:-1], -1, bits_per_index) * weights).sum(-1)

In [ ]:
x = np.array([[0, 1, 5, 255], [128, 7, 2, 3]])
b = to_bits(x, 8)
assert b.shape == (2, 32) and b[0, 8:16].tolist() == [1, 0, 0, 0, 0, 0, 0, 0]       # 1, least significant bit first
assert (from_bits(b, 8) == x).all() and (from_bits(b.astype(bool), 8) == x).all()

## Codecs

A `Codec` turns a payload into bits and back. Discrete messages and continuous ones each have theirs:

| payload | codec | bit errors give |
|---|---|---|
| discrete tokens, e.g. a VQ-VAE's codebook indices | `IndexCodec` | other tokens |
| continuous values, e.g. a latent vector of 128 numbers | `UniformQuantizer`: each number on `bits` bits (scalar quantization) | values off by up to the range (most significant bit) |
| continuous vectors, quantized together | `VectorQuantizer`: the nearest codeword of a codebook, as its index (vector quantization) | other codewords |

Protecting the bits against the channel's errors is channel coding, a different stage (see `coding`).

In [ ]:
#| export
class Codec(ABC):
    "Payload <-> bits."

    @abstractmethod
    def encode(self, payload):
        "The payload's bits: (..., num_bits) of 0/1."

    @abstractmethod
    def decode(self, bits):
        "The payload of `bits`, which may have bit errors."


class IndexCodec(Codec):
    "Indices into a codebook (e.g. a VQ-VAE's), with ceil(log2(codebook_size)) bits each."

    def __init__(
        self,
        codebook_size: int = 256, # Number of codewords
    ):
        self.codebook_size = codebook_size
        self.bits_per_index = max(1, math.ceil(math.log2(codebook_size)))

    def encode(self, payload):
        "`payload`: integers in [0, codebook_size), (..., M)."
        x = _as_array(payload)
        if _is_torch(x):
            integer = not (x.is_floating_point() or x.is_complex()) and str(x.dtype) != 'torch.bool'
        else:
            integer = np.issubdtype(x.dtype, np.integer)
        if not integer:
            raise TypeError(f"IndexCodec encodes integers, not {x.dtype}")
        if (x < 0).any() or (x >= self.codebook_size).any():
            raise ValueError(f"indices must be in [0, {self.codebook_size})")
        return to_bits(x, self.bits_per_index)

    def decode(self, bits):
        "Indices (..., M). Bit errors can give indices >= codebook_size if it is not a power of 2."
        return from_bits(bits, self.bits_per_index)


class UniformQuantizer(Codec):
    "Real numbers in [low, high], each rounded to one of 2**bits evenly spaced levels (values outside are clipped)."

    def __init__(
        self,
        bits: int = 8, # Bits per number
        low: float = -1.0, # Smallest value
        high: float = 1.0, # Largest value
    ):
        self.bits, self.low, self.high = bits, low, high
        self.levels = 2 ** bits - 1

    def encode(self, payload):
        x = _as_array(payload, dtype=float)
        q = ((x.clip(self.low, self.high) - self.low) / (self.high - self.low) * self.levels).round()
        return to_bits(q.long() if _is_torch(q) else q.astype(np.int64), self.bits)

    def decode(self, bits):
        return self.low + from_bits(bits, self.bits) * ((self.high - self.low) / self.levels)


class VectorQuantizer(Codec):
    "Vectors, each replaced by the nearest codeword of a codebook and sent as its index (ceil(log2(size)) bits)."

    def __init__(
        self,
        codebook, # The codewords, (C, d): e.g. a VQ-VAE's embeddings
    ):
        self.codebook = codebook
        self.index = IndexCodec(len(codebook))
        self.bits_per_index = self.index.bits_per_index

    def quantize(
        self,
        payload, # Vectors, (..., d)
    ):
        "The index of each vector's nearest codeword, (...)."
        x = _as_array(payload, dtype=float)
        if _is_torch(x):
            import torch
            book = torch.as_tensor(self.codebook, dtype=x.dtype, device=x.device)
            return ((x[..., None, :] - book) ** 2).sum(-1).argmin(-1)
        return ((x[..., None, :] - np.asarray(self.codebook, float)) ** 2).sum(-1).argmin(-1)

    def encode(self, payload):
        "`payload`: vectors, (..., M, d) -> bits, (..., M * bits_per_index)."
        return self.index.encode(self.quantize(payload))

    def decode(self, bits):
        "The codewords, (..., M, d). Bit errors that give an index past the codebook give its last codeword."
        i = self.index.decode(bits)
        if _is_torch(i):
            import torch
            return torch.as_tensor(self.codebook, device=i.device)[i.clamp(max=len(self.codebook) - 1)]
        return np.asarray(self.codebook)[np.minimum(i, len(self.codebook) - 1)]

In [ ]:
c = IndexCodec(256)
assert c.bits_per_index == 8 and c.encode([3, 200]).shape == (16,) and c.decode(c.encode([3, 200])).tolist() == [3, 200]
assert IndexCodec(200).bits_per_index == 8 and IndexCodec(2).bits_per_index == 1
for bad, err in (([1.5], TypeError), ([True], TypeError), ([256], ValueError), ([-1], ValueError)):
    try:
        c.encode(bad); raise AssertionError(bad)
    except err:
        pass
q = UniformQuantizer(bits=8, low=-1, high=1)
x = np.array([-1.0, -0.3, 0.0, 0.77, 1.0, 3.0])
y = q.decode(q.encode(x))
assert y.shape == x.shape and np.abs(y - x.clip(-1, 1)).max() <= 1 / 255 + 1e-12      # within half a step (2/255 / 2)
# a latent vector of 128 numbers: 8 bits each
latent = np.random.default_rng(0).standard_normal(128).clip(-3, 3)
q3 = UniformQuantizer(bits=8, low=-3, high=3)
assert q3.encode(latent).shape == (1024,) and np.abs(q3.decode(q3.encode(latent)) - latent).max() <= 3 / 255 + 1e-12
# vectors: the nearest of 6 codewords in 2-D, 3 bits each; an index past the codebook gives the last codeword
vq = VectorQuantizer(np.array([[0, 0], [1, 0], [0, 1], [1, 1], [2, 2], [-1, -1]], float))
v = np.array([[0.9, 0.2], [1.8, 2.3], [-0.6, -0.9]])
assert vq.quantize(v).tolist() == [1, 4, 5] and vq.encode(v).shape == (9,)
assert vq.decode(vq.encode(v)).tolist() == [[1, 0], [2, 2], [-1, -1]]
assert vq.decode(to_bits(np.array([7]), 3)).tolist() == [[-1, -1]]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()